In [148]:
import requests
import pandas as pd
from bs4 import BeautifulSoup as bs
from io import StringIO
from datetime import date
from pathlib import Path

## Células de teste:

In [149]:
url = "https://pt.wikipedia.org/wiki/Marvel_Games"
#COLOQUE O SEU E-MAIL NO CAMPO ABAIXO (substitua-o)!!!
email = "guilhermebronzeado09@gmail.com"
headers = {"User-Agent": f"projeto-faculdade-dc-vs-marvel/1.0 ({email})"}

page = requests.get(url, headers=headers, timeout = 30)
page.raise_for_status()
print(page.status_code)

200


In [ ]:
soup = bs(page.text, "html.parser")
tabelas = soup.find_all("table", class_="wikitable")
print(len(tabelas))

In [ ]:
tabela = tabelas[0]
cabecalho = [th.text.strip() for th in tabela.find_all("th")]
print(cabecalho)

linhas = tabela.find_all("tr")
for linha in linhas[:12]:
    celulas = [td.text.strip() for td in linha.find_all("td")]
    if not celulas:
        continue
    print(len(celulas), celulas)

In [ ]:
#Verifica quais colunas/tabelas estão juntas pelo html da wikipedia
# (o que ocasiona o erro de len = 4 nas celulas anteriores)
for linha in linhas[:12]:
    for td in linha.find_all("td"):
        if td.get("rowspan") or td.get("colspan"):
            print(td.text.strip(), "-> rowspan:", td.get("rowspan"), "| colspan:", td.get("colspan"))


In [ ]:
df_teste = pd.read_html(StringIO(str(tabela)))[0]
print(df_teste.shape)
df_teste.head(12)

In [ ]:
quadros = []

for tabela in tabelas:
    titulo = tabela.find_previous(["h2", "h3"])
    secao = titulo.text.strip() if titulo else None

    df_parcial = pd.read_html(StringIO(str(tabela)))[0]
    df_parcial["secao_wikipedia"] = secao
    quadros.append(df_parcial)

df_dc = pd.concat(quadros, ignore_index=True)
print(df_dc.shape)
df_dc.head()

In [ ]:
df_dc["url_fonte"] = url
df_dc["data_coleta"] = date.today().isoformat()

pasta = Path("../data/raw")
pasta.mkdir(parents=True, exist_ok=True)
df_dc.to_csv(pasta / "wikipedia_jogos_marvel.csv", index=False, encoding="utf-8")

In [ ]:
print(df_dc.shape)
print(df_dc.isna().sum())
print(df_dc["secao_wikipedia"].value_counts())
df_dc[df_dc.iloc[:, 0] == df_dc.iloc[:, 1]]   # linhas onde a 1ª e a 2ª coluna são iguais

## Função que cria o csv:

In [ ]:
def coletar_lista(url, universo):
    
    page = requests.get(url, headers=headers, timeout=30)
    page.raise_for_status()

    soup = bs(page.text, "html.parser")
    tabelas = soup.find_all("table", class_="wikitable")
    print(f"{universo}: {len(tabelas)} tabelas encontradas")

    quadros = []
    for tabela in tabelas:
        titulo = tabela.find_previous(["h2", "h3"])
        secao = titulo.text.strip() if titulo else None

        df_parcial = pd.read_html(StringIO(str(tabela)))[0]
        df_parcial["secao_wikipedia"] = secao
        quadros.append(df_parcial)

    df = pd.concat(quadros, ignore_index=True)
    df["url_fonte"] = url
    df["data_coleta"] = date.today().isoformat()

    pasta = Path("../data/raw")
    pasta.mkdir(parents=True, exist_ok=True)
    df.to_csv(pasta / f"wikipedia_jogos_{universo}.csv", index=False, encoding="utf-8")
    return df

## Chamada da coleta das informações 
(Usa o pandas e o BeautifulSoup para tratar o código html, montar um data_frame e formata-lo num csv)

In [ ]:
#COLOQUE O SEU E-MAIL NO CAMPO ABAIXO (substitua-o)!!!
email = "guilhermebronzeado09@gmail.com"
headers = {"User-Agent": f"projeto-faculdade-dc-vs-marvel/1.0 ({email})"}

In [150]:
URL_DC = "https://en.wikipedia.org/wiki/List_of_video_games_based_on_DC_Comics"
df_dc = coletar_lista(URL_DC, "dc")

dc: 5 tabelas encontradas


In [151]:
URL_MARVEL = "https://pt.wikipedia.org/wiki/Marvel_Games"
df_marvel = coletar_lista(URL_MARVEL, "marvel")

marvel: 5 tabelas encontradas
